# Modeling & Evaluation

This notebook trains baseline and advanced models for credit risk prediction and evaluates them using ROC-AUC and confusion matrices.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns

**Note:** This notebook assumes that the preprocessed matrices
`X_train_processed`, `X_test_processed`, `y_train`, and `y_test`
are available from the preprocessing step.

## Baseline Model: Logistic Regression

In [ ]:
log_model = LogisticRegression(max_iter=1000, class_weight='balanced')
log_model.fit(X_train_processed, y_train)

train_pred_lr = log_model.predict_proba(X_train_processed)[:,1]
test_pred_lr = log_model.predict_proba(X_test_processed)[:,1]

print('Train ROC-AUC:', roc_auc_score(y_train, train_pred_lr))
print('Test ROC-AUC:', roc_auc_score(y_test, test_pred_lr))

## XGBoost Model

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    random_state=42
)

xgb_model.fit(X_train_processed, y_train)

In [ ]:
test_pred_xgb = xgb_model.predict_proba(X_test_processed)[:,1]
print('XGBoost Test ROC-AUC:', roc_auc_score(y_test, test_pred_xgb))

In [ ]:
threshold = 0.4
y_pred = (test_pred_xgb >= threshold).astype(int)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d')
plt.title('Confusion Matrix - XGBoost')
plt.show()

XGBoost outperforms the baseline and is selected as the final model.